####DAY 10 (01/03/26) – Query Optimization & Explain Plans
####🏗️ Architecture & Strategy
Welcome to Phase 3: Performance & Production Thinking! You have built models and pipelines; now it is time to make them run blazingly fast and cost-effectively. Today, we are diving deep into the engine room of Apache Spark: The Catalyst Optimizer .

When you write PySpark code, Spark doesn't just execute it line-by-line. The Catalyst Optimizer analyzes your code, rewrites it for maximum efficiency, and generates a "Physical Execution Plan."

####Our Senior-Level Strategy:

* **The "Heavy" Query**: We will run a heavy aggregation on our massive 56-million row events_delta_managed table.

* **Reading the Explain Plan**: We will use .explain(True) to peek under the hood and prove that Spark is using "Predicate Pushdown" (filtering data at the disk level before loading it into RAM).

* **Execution Time Baseline**: We will measure exactly how long this query takes to run cold.

* **Caching & The Serverless Reality**: The challenge asks us to enable caching and compare execution times. Because you are running on Databricks Serverless (as we discovered on Day 9), traditional Spark .cache() is explicitly blocked to allow Databricks' intelligent Data Cache to take over. I have written a dynamic script that will attempt traditional caching, elegantly catch the Serverless restriction, and explain how modern Data Lakehouses handle performance .

####Analyze Explain Plan & Predicate Pushdown
Let's define a heavy query that calculates the total purchases for every product, and ask Spark to explain how it plans to compute it.

In [0]:
from pyspark.sql import functions as F

# 1. Environment Setup
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print("⏳ Connecting to the massive events table...")
events_df = spark.table("events_delta_managed")

# 2. Define a "Heavy" Query
# We filter for purchases, group by product, count them, and sort.
heavy_query_df = (
    events_df
    .filter(F.col("event_type") == "purchase")
    .groupBy("product_id")
    .agg(F.count("*").alias("total_purchases"))
    .orderBy(F.col("total_purchases").desc())
)

# 3. Analyze the Explain Plan
print("🧠 Catalyst Optimizer Execution Plan:")
# extended=True gives us the Logical, Optimized Logical, and Physical plans.
# Focus on the "Physical Plan" at the very bottom of the output!
heavy_query_df.explain(extended=True)

# 💡 HOW TO READ THE OUTPUT (Scroll to the bottom of the output):
# 1. Look for 'FileScan parquet' -> This shows Spark reading the Delta table.
# 2. Look for 'PushedFilters: [IsNotNull(event_type), (event_type = purchase)]'. 
#    This proves Predicate Pushdown is working! Spark isn't loading all 56M rows; 
#    it is only loading rows where event_type='purchase' directly from the disk.

####Baseline Execution Time (Cold Run)
Now, let's trigger an action (.count()) to force Spark to execute the plan and see how long it takes without any memory assistance.

In [0]:
import time

print("⏱️ Running heavy query without caching (Cold Run)...")

start_time = time.time()

# We call .count() to trigger the lazy evaluation and force computation
total_unique_products = heavy_query_df.count() 

end_time = time.time()
cold_run_time = end_time - start_time

print(f"   ➤ Total unique purchased products: {total_unique_products:,}")
print(f"   🔴 Execution Time (Cold): {cold_run_time:.2f} seconds")

####Enable Caching & Compare Execution Time
In a traditional Spark cluster, we would use .cache() to store the base table in cluster RAM. Let's execute the caching strategy required by the task, while handling the advanced Serverless architecture restrictions you are working within.

In [0]:
print("⚙️ Attempting to enable Spark Memory Caching and re-run query...")

try:
    # 1. Attempt Traditional Spark Caching
    # This stores the 56M row table in the worker nodes' RAM.
    events_df.cache()
    
    # Force the cache to materialize (load into memory)
    events_df.count() 
    
    # 2. Re-run the heavy query using the cached memory
    start_time = time.time()
    heavy_query_df.count()
    end_time = time.time()
    
    cached_run_time = end_time - start_time
    
    print(f"   🟢 Execution Time (Cached): {cached_run_time:.2f} seconds")
    print(f"   🚀 Speedup: {cold_run_time / cached_run_time:.1f}x faster!")
    
    # Always clear cache to free up cluster resources when done
    events_df.unpersist()

except Exception as e:
    # 3. Handle Databricks Serverless Auto-Caching
    # If you see this block execute, it proves you are on Serverless!
    print("\n🛡️ SERVERLESS ARCHITECTURE DETECTED!")
    print("   Databricks blocked traditional .cache() because Serverless uses an advanced,")
    print("   fully-managed SSD Data Cache behind the scenes.")
    
    # Even without explicit .cache(), the second run will be faster because 
    # the Serverless engine cached the underlying parquet files on NVMe SSDs during Cell 2!
    print("\n⏱️ Re-running query relying on Serverless Auto-Caching...")
    
    start_time = time.time()
    heavy_query_df.count()
    end_time = time.time()
    
    serverless_cached_time = end_time - start_time
    print(f"   🟢 Execution Time (Serverless Cache): {serverless_cached_time:.2f} seconds")
    
    if cold_run_time > 0 and serverless_cached_time > 0:
        print(f"   🚀 Auto-Speedup: {cold_run_time / serverless_cached_time:.1f}x faster without writing a single line of caching code!")

####Key Learnings & Interview Talking Points
If a technical recruiter asks about how you optimize PySpark pipelines for performance, use these critical points:

* **The Catalyst Optimizer & `.explain()`**: "I never blindly run PySpark code. I use `.explain(True)` to audit the physical execution plan. This allows me to verify that my code is triggering efficient operations like HashAggregates instead of expensive SortAggregates."

* **Predicate Pushdown**: "When analyzing explain plans, I actively look for `PushedFilters`. This proves that my filtering logic (like `WHERE event_type='purchase'`) is being pushed down to the Parquet/Delta storage layer, drastically reducing the amount of data serialized into cluster memory over the network."

* **Strategic Caching & Serverless Evolution**: "On traditional clusters, I use `.cache()` to persist heavily queried DataFrames into RAM, allowing subsequent queries to execute orders of magnitude faster. However, I am also experienced with modern Databricks Serverless architectures. On Serverless, explicit memory management is intentionally restricted because Databricks relies on an automatic, highly optimized NVMe SSD Data Cache, which accelerates repeated queries without requiring manual `unpersist()` cleanup.